# Experimento: VCP Pipeline Completo con Todos los Filtros

Pipeline VCP completo con **todos los filtros configurables**:

| Filtro | Parametro de control | Default |
|--------|---------------------|---------|
| Trend Template (Etapa 2) | `USE_TREND_TEMPLATE` | `True` |
| Ascending Lows (lows ascendentes) | `SEQUENCE_PARAMS["require_ascending_lows"]` | `True` |
| Max Entry Distance (distancia al pivote) | `BREAKOUT_PARAMS["max_entry_distance_pct"]` | `0.05` |
| Volume Confirmation (volumen en breakout) | `BREAKOUT_PARAMS["require_volume_confirmation"]` | `True` |
| Max Depth % (profundidad maxima absoluta) | `SEQUENCE_PARAMS["max_depth_pct"]` | `0.35` |
| **Max Depth ATR (profundidad relativa)** | `SEQUENCE_PARAMS["max_depth_atr"]` | `5.0` |
| Min Total Reduction (compresion real) | `SEQUENCE_PARAMS["min_total_reduction"]` | `0.80` |
| Volume Contraction | `USE_VOLUME_CONTRACTION` | `True` |

Cada filtro puede activarse/desactivarse independientemente.
Los resultados se registran en MLflow con metricas de drawdown.

In [1]:
import sys
import tempfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.configs import ATRZigZagConfig
from vcp_detection.heuristic import ATRZigZagDetector, run_full_vcp_pipeline
from vcp_detection.analysis import (
    group_signals_into_patterns,
    simulate_trade,
    plot_vcp_pattern,
    plot_trade_simulation,
)
from stages.trend_template import evaluate_trend_template

pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")

/home/gdelarosa/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## 1. Configuracion

Todos los filtros se controlan desde esta celda. Para desactivar un filtro:
- **Trend Template**: `USE_TREND_TEMPLATE = False`
- **Ascending Lows**: `require_ascending_lows: False` en SEQUENCE_PARAMS
- **Max Entry Distance**: `max_entry_distance_pct: None` en BREAKOUT_PARAMS
- **Max Depth ATR**: `max_depth_atr: None` en SEQUENCE_PARAMS
- **Volume Contraction**: `USE_VOLUME_CONTRACTION = False`
- **Quality filters**: `max_depth_pct: None` o `min_total_reduction: None`

In [ ]:
# ============================================================================
# FILTROS TOGGLEABLES
# ============================================================================
USE_TREND_TEMPLATE =  False
USE_VOLUME_CONTRACTION = True

# ============================================================================
# GRILLA DE EXPERIMENTO
# ============================================================================
VOLUME_RATIO_THRESHOLDS = [1.0, 1.5, 2.0]

# ============================================================================
# PARAMETROS DEL PIPELINE
# ============================================================================
SWING_CONFIG = ATRZigZagConfig(
    atr_length=14,
    atr_mult=2.0,
    use_close_only=False,
)

SEQUENCE_PARAMS = {
    "method": "tolerance",
    "min_contractions": 2,
    "max_contractions": 6,
    "lookback_bars": 126,
    "tolerance": 0.10,
    "max_depth_pct": 0.35,
    "max_depth_atr": 7,
    "min_total_reduction": 0.80,
    "max_gap_between_contractions_days": None,
    # --- Ascending Lows ---
    "require_ascending_lows": True,
    "ascending_lows_tolerance": 0.10,
}

COMPRESSION_PARAMS = {
    "method": "ratio",
    "atr_period": 14,
    "ratio_threshold": 0.85,
}

VOLUME_CONTRACTION_PARAMS = {
    "method": "ratio",
    "volume_column": "volume",
    "ratio_threshold": 0.85,
}

BREAKOUT_PARAMS = {
    "volume_method": "ratio",
    "volume_ratio_threshold": 1.5,  # se sobreescribe por la grilla
    "volume_lookback_days": 50,
    "require_volume_confirmation": True,
    # --- Max Entry Distance ---
    "max_entry_distance_pct": 0.10,
}

RISK_PARAMS = {
    "max_stop_loss_pct": 0.05,
    "breakeven_r_multiple": 2.0,
    "trailing_sma_period": 20,
    "trailing_volume_factor": 1.5,
    # --- Trailing Stop Method: "sma" (original) o "atr" (adaptativo) ---
    "trailing_stop_method": "atr",
    "trailing_atr_period": 14,
    "trailing_atr_multiplier": 3.0,
    # --- Time Exit: None para desactivar, int para max bars sin progreso ---
    "max_bars_without_progress": 20,
    "min_progress_r": 0.5,
    # --- Early Exit (Minervini): salir si close < entry en los primeros N dias ---
    "early_exit_days": 3,
}

# ============================================================================
# TICKERS
# ============================================================================
DATA_DIR = project_root / "data" / "csv"
TICKERS = sorted([p.stem for p in DATA_DIR.glob("*.csv")])

# ============================================================================
# RESUMEN DE CONFIGURACION
# ============================================================================
print(f"Tickers ({len(TICKERS)}): {TICKERS}")
print(f"Grilla: volume_ratio_threshold = {VOLUME_RATIO_THRESHOLDS}")
print(f"Total corridas: {len(VOLUME_RATIO_THRESHOLDS)} x {len(TICKERS)} = {len(VOLUME_RATIO_THRESHOLDS) * len(TICKERS)}")
print()
print("Filtros activos:")
print(f"  Trend Template:        {USE_TREND_TEMPLATE}")
print(f"  Ascending Lows:        {SEQUENCE_PARAMS['require_ascending_lows']} (tol={SEQUENCE_PARAMS['ascending_lows_tolerance']})")
print(f"  Max Entry Distance:    {BREAKOUT_PARAMS['max_entry_distance_pct']}")
print(f"  Volume Confirmation:   {BREAKOUT_PARAMS['require_volume_confirmation']}")
print(f"  Volume Contraction:    {USE_VOLUME_CONTRACTION}")
print(f"  Max Depth %:           {SEQUENCE_PARAMS['max_depth_pct']}")
print(f"  Max Depth ATR:         {SEQUENCE_PARAMS['max_depth_atr']}")
print(f"  Min Total Reduction:   {SEQUENCE_PARAMS['min_total_reduction']}")
print()
print("Gestion de trade:")
print(f"  Trailing Stop Method:  {RISK_PARAMS['trailing_stop_method']}")
if RISK_PARAMS["trailing_stop_method"] == "atr":
    print(f"    ATR Period:          {RISK_PARAMS['trailing_atr_period']}")
    print(f"    ATR Multiplier:      {RISK_PARAMS['trailing_atr_multiplier']}")
print(f"  Max Stop Loss:         {RISK_PARAMS['max_stop_loss_pct']:.0%}")
print(f"  Early Exit:            {RISK_PARAMS['early_exit_days']} dias" if RISK_PARAMS.get("early_exit_days") else "  Early Exit:            OFF")
print(f"  Time Exit:             {RISK_PARAMS['max_bars_without_progress']} bars (min_progress={RISK_PARAMS['min_progress_r']}R)" if RISK_PARAMS["max_bars_without_progress"] else "  Time Exit:             OFF")

## 2. Funciones auxiliares

In [15]:
def load_ohlc(ticker: str) -> pd.DataFrame:
    path = DATA_DIR / f"{ticker}.csv"
    return pd.read_csv(path, parse_dates=["date"], index_col="date")


def compute_trade_max_drawdown(ohlc: pd.DataFrame, entry_date, exit_date) -> float:
    """Max drawdown durante un trade."""
    trade_closes = ohlc.loc[entry_date:exit_date, "close"]
    if len(trade_closes) < 2:
        return 0.0
    running_peak = trade_closes.cummax()
    drawdown_series = (trade_closes - running_peak) / running_peak
    return float(drawdown_series.min())


def compute_asset_max_drawdown(ohlc: pd.DataFrame) -> float:
    """Max drawdown del activo en todo el periodo."""
    close = ohlc["close"]
    running_peak = close.cummax()
    drawdown_series = (close - running_peak) / running_peak
    return float(drawdown_series.min())


def compute_in_trade_sharpe(ohlc: pd.DataFrame, trades: list[dict], annualization: float = 252.0) -> float:
    """Sharpe ratio usando solo los retornos diarios durante trades activos."""
    daily_returns = []
    for t in trades:
        entry_date = t["pattern"]["first_signal_date"]
        exit_date = t["exit_date"]
        trade_closes = ohlc.loc[entry_date:exit_date, "close"]
        if len(trade_closes) >= 2:
            rets = trade_closes.pct_change().dropna()
            daily_returns.append(rets)
    if not daily_returns:
        return 0.0
    all_rets = pd.concat(daily_returns)
    if len(all_rets) < 2 or all_rets.std() == 0:
        return 0.0
    return float(all_rets.mean() / all_rets.std() * np.sqrt(annualization))


def run_ticker_analysis(
    ticker: str,
    ohlc: pd.DataFrame,
    template_df: pd.DataFrame | None,
    breakout_params: dict,
    risk_params: dict,
) -> dict:
    """Pipeline VCP completo con todos los filtros configurables."""
    swing_detector = ATRZigZagDetector(SWING_CONFIG)

    vol_params = VOLUME_CONTRACTION_PARAMS if USE_VOLUME_CONTRACTION else None

    results = run_full_vcp_pipeline(
        ohlc=ohlc,
        swing_detector=swing_detector,
        sequence_params=SEQUENCE_PARAMS,
        compression_params=COMPRESSION_PARAMS,
        breakout_params=breakout_params,
        volume_contraction_params=vol_params,
    )

    signals_raw = {dt: sig for dt, sig in results.items() if sig is not None}

    all_signals = {}
    n_filtered_template = 0
    if USE_TREND_TEMPLATE and template_df is not None:
        for dt, sig in signals_raw.items():
            if dt in template_df.index and bool(template_df.loc[dt, "trend_template"]):
                all_signals[dt] = sig
            else:
                n_filtered_template += 1
    else:
        all_signals = dict(signals_raw)

    patterns = group_signals_into_patterns(all_signals, risk_params=risk_params)

    trades = []
    for p in patterns:
        trade = simulate_trade(ohlc, p, risk_params)
        trade["pattern"] = p
        trade["max_drawdown"] = compute_trade_max_drawdown(
            ohlc, p["first_signal_date"], trade["exit_date"],
        )
        trades.append(trade)

    asset_max_dd = compute_asset_max_drawdown(ohlc)
    sharpe_ratio = compute_in_trade_sharpe(ohlc, trades)

    n_trades = len(trades)
    if n_trades > 0:
        wins = sum(1 for t in trades if t["pnl_pct"] > 0)
        winrate = wins / n_trades
        cumulative_return = float(np.prod([1 + t["pnl_pct"] for t in trades]) - 1)
        avg_r_multiple = float(np.mean([t["r_multiple"] for t in trades]))
        worst_trade_dd = float(min(t["max_drawdown"] for t in trades))
        avg_trade_dd = float(np.mean([t["max_drawdown"] for t in trades]))
        avg_entry_distance = float(np.mean([
            t["pattern"].get("metadata", {}).get("entry_distance_pct", 0.0)
            for t in trades
        ])) if trades else 0.0
    else:
        winrate = 0.0
        cumulative_return = 0.0
        avg_r_multiple = 0.0
        worst_trade_dd = 0.0
        avg_trade_dd = 0.0
        avg_entry_distance = 0.0

    return {
        "signals": all_signals,
        "patterns": patterns,
        "trades": trades,
        "metrics": {
            "n_signals_raw": len(signals_raw),
            "n_signals": len(all_signals),
            "n_filtered_by_template": n_filtered_template,
            "n_patterns": len(patterns),
            "n_trades": n_trades,
            "winrate": winrate,
            "cumulative_return": cumulative_return,
            "avg_r_multiple": avg_r_multiple,
            "worst_trade_drawdown": worst_trade_dd,
            "avg_trade_drawdown": avg_trade_dd,
            "asset_max_drawdown": asset_max_dd,
            "avg_entry_distance": avg_entry_distance,
            "sharpe_ratio": sharpe_ratio,
        },
    }


def build_trade_table(trades: list[dict], ticker: str) -> pd.DataFrame:
    rows = []
    for i, t in enumerate(trades, 1):
        entry_dist = t["pattern"].get("metadata", {}).get("entry_distance_pct", None)
        rows.append({
            "ticker": ticker,
            "trade_num": i,
            "entry_date": t["pattern"]["first_signal_date"].strftime("%Y-%m-%d"),
            "exit_date": t["exit_date"].strftime("%Y-%m-%d"),
            "exit_reason": t["exit_reason"],
            "duration_days": t["duration_days"],
            "entry_price": t["pattern"]["entry_price"],
            "exit_price": t["exit_price"],
            "pnl_pct": t["pnl_pct"],
            "r_multiple": t["r_multiple"],
            "max_r": t["max_r"],
            "max_drawdown": t["max_drawdown"],
            "entry_distance_pct": entry_dist,
            "stop_method": t["pattern"]["stop_method"],
            "n_contractions": t["pattern"]["n_contractions"],
            "atr_ratio": t["pattern"]["atr_ratio"],
        })
    return pd.DataFrame(rows)


print("Funciones auxiliares definidas.")

Funciones auxiliares definidas.


## 3. Setup MLflow

In [16]:
EXPERIMENT_NAME = "VCP_FullFilters"
mlflow.set_tracking_uri(str(project_root / "mlruns"))
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

MLflow experiment: VCP_FullFilters
Tracking URI: /home/gdelarosa/proyectos/deteccion-vcp/mlruns


## 4. Pre-computo: Trend Template por ticker

Se evalua una sola vez (no depende del threshold). Si `USE_TREND_TEMPLATE = False`, se saltea.

In [17]:
template_cache = {}

if USE_TREND_TEMPLATE:
    template_summary = []
    for ticker in TICKERS:
        ohlc = load_ohlc(ticker)
        tpl = evaluate_trend_template(ohlc)
        template_cache[ticker] = tpl

        n_days = len(tpl)
        n_stage2 = int(tpl["trend_template"].sum())
        pct = n_stage2 / n_days if n_days > 0 else 0.0
        template_summary.append({
            "ticker": ticker,
            "total_days": n_days,
            "days_in_stage2": n_stage2,
            "pct_in_stage2": pct,
        })

    template_summary_df = pd.DataFrame(template_summary).set_index("ticker")
    print("Trend Template evaluado para todos los tickers:\n")
    display(template_summary_df.style.format({
        "pct_in_stage2": "{:.1%}",
    }).background_gradient(subset=["pct_in_stage2"], cmap="RdYlGn", vmin=0, vmax=1))
else:
    print("Trend Template DESACTIVADO - no se pre-computa.")

Trend Template DESACTIVADO - no se pre-computa.


## 5. Ejecucion del experimento

Por cada `volume_ratio_threshold` se crea un **parent run** en MLflow.
Dentro, por cada ticker un **child run** con metricas, plots, tablas y drawdown.

In [ ]:
all_summaries = []

ALL_PARAMS = {
    # Swing
    "atr_length": SWING_CONFIG.atr_length,
    "atr_mult": SWING_CONFIG.atr_mult,
    "use_close_only": SWING_CONFIG.use_close_only,
    # Sequence
    "seq_method": SEQUENCE_PARAMS["method"],
    "min_contractions": SEQUENCE_PARAMS["min_contractions"],
    "max_contractions": SEQUENCE_PARAMS["max_contractions"],
    "lookback_bars": SEQUENCE_PARAMS["lookback_bars"],
    "tolerance": SEQUENCE_PARAMS["tolerance"],
    "max_depth_pct": str(SEQUENCE_PARAMS["max_depth_pct"]),
    "max_depth_atr": str(SEQUENCE_PARAMS.get("max_depth_atr")),
    "min_total_reduction": str(SEQUENCE_PARAMS["min_total_reduction"]),
    "max_gap_days": str(SEQUENCE_PARAMS["max_gap_between_contractions_days"]),
    # Ascending Lows
    "require_ascending_lows": SEQUENCE_PARAMS["require_ascending_lows"],
    "ascending_lows_tolerance": SEQUENCE_PARAMS["ascending_lows_tolerance"],
    # Compression
    "compression_method": COMPRESSION_PARAMS["method"],
    "compression_threshold": COMPRESSION_PARAMS["ratio_threshold"],
    # Volume Contraction
    "use_volume_contraction": USE_VOLUME_CONTRACTION,
    "vol_contraction_method": VOLUME_CONTRACTION_PARAMS["method"],
    "vol_contraction_threshold": VOLUME_CONTRACTION_PARAMS["ratio_threshold"],
    # Breakout
    "max_entry_distance_pct": str(BREAKOUT_PARAMS["max_entry_distance_pct"]),
    # Risk
    "max_stop_loss_pct": RISK_PARAMS["max_stop_loss_pct"],
    "breakeven_r_multiple": RISK_PARAMS["breakeven_r_multiple"],
    "trailing_sma_period": RISK_PARAMS["trailing_sma_period"],
    "trailing_volume_factor": RISK_PARAMS["trailing_volume_factor"],
    "trailing_stop_method": RISK_PARAMS.get("trailing_stop_method", "sma"),
    "trailing_atr_period": RISK_PARAMS.get("trailing_atr_period", 14),
    "trailing_atr_multiplier": RISK_PARAMS.get("trailing_atr_multiplier", 3.0),
    "max_bars_without_progress": str(RISK_PARAMS.get("max_bars_without_progress")),
    "min_progress_r": RISK_PARAMS.get("min_progress_r", 0.5),
    "early_exit_days": str(RISK_PARAMS.get("early_exit_days")),
    # Trend Template
    "use_trend_template": USE_TREND_TEMPLATE,
}

for threshold in VOLUME_RATIO_THRESHOLDS:
    breakout_params = {
        **BREAKOUT_PARAMS,
        "volume_ratio_threshold": threshold,
    }

    config_name = f"threshold_{threshold}"
    print(f"\n{'='*70}")
    print(f"CONFIG: {config_name} (volume_ratio_threshold={threshold})")
    print(f"{'='*70}")

    with mlflow.start_run(run_name=config_name) as parent_run:
        mlflow.log_params({
            **ALL_PARAMS,
            "volume_ratio_threshold": threshold,
        })

        ticker_summaries = []
        all_trades_for_config = []

        for ticker in TICKERS:
            print(f"  {ticker}...", end=" ")
            ohlc = load_ohlc(ticker)
            template_df = template_cache.get(ticker)

            with mlflow.start_run(run_name=ticker, nested=True) as child_run:
                mlflow.log_params({
                    **ALL_PARAMS,
                    "ticker": ticker,
                    "volume_ratio_threshold": threshold,
                    "n_bars": len(ohlc),
                })

                analysis = run_ticker_analysis(
                    ticker, ohlc, template_df, breakout_params, RISK_PARAMS,
                )
                metrics = analysis["metrics"]

                n_wins = sum(1 for t in analysis["trades"] if t["pnl_pct"] > 0)
                n_losses = metrics["n_trades"] - n_wins

                mlflow.log_metrics({
                    "n_signals_raw": metrics["n_signals_raw"],
                    "n_signals": metrics["n_signals"],
                    "n_filtered_by_template": metrics["n_filtered_by_template"],
                    "n_patterns": metrics["n_patterns"],
                    "n_trades": metrics["n_trades"],
                    "n_wins": n_wins,
                    "n_losses": n_losses,
                    "winrate": metrics["winrate"],
                    "cumulative_return": metrics["cumulative_return"],
                    "avg_r_multiple": metrics["avg_r_multiple"],
                    "worst_trade_drawdown": metrics["worst_trade_drawdown"],
                    "avg_trade_drawdown": metrics["avg_trade_drawdown"],
                    "asset_max_drawdown": metrics["asset_max_drawdown"],
                    "avg_entry_distance": metrics["avg_entry_distance"],
                    "sharpe_ratio": metrics["sharpe_ratio"],
                })

                with tempfile.TemporaryDirectory() as tmpdir:
                    for j, pattern in enumerate(analysis["patterns"], 1):
                        plot_path = Path(tmpdir) / f"pattern_{j}.png"
                        plot_vcp_pattern(
                            ohlc, pattern, pattern_number=j,
                            ticker=ticker, save_path=str(plot_path),
                        )
                        mlflow.log_artifact(str(plot_path), "pattern_plots")

                    for j, (pat, trade) in enumerate(
                        zip(analysis["patterns"], analysis["trades"]), 1
                    ):
                        trade_plot_path = Path(tmpdir) / f"trade_{j}.png"
                        plot_trade_simulation(
                            ohlc, pat, trade, pattern_number=j,
                            risk_params=RISK_PARAMS, ticker=ticker,
                            save_path=str(trade_plot_path),
                        )
                        mlflow.log_artifact(str(trade_plot_path), "trade_plots")

                    if analysis["trades"]:
                        trade_df = build_trade_table(analysis["trades"], ticker)
                        trade_csv_path = Path(tmpdir) / "trades.csv"
                        trade_df.to_csv(trade_csv_path, index=False)
                        mlflow.log_artifact(str(trade_csv_path), "tables")
                        all_trades_for_config.append(trade_df)

                ticker_summaries.append({
                    "ticker": ticker,
                    "n_signals_raw": metrics["n_signals_raw"],
                    "n_signals": metrics["n_signals"],
                    "n_filtered": metrics["n_filtered_by_template"],
                    "n_patterns": metrics["n_patterns"],
                    "n_trades": metrics["n_trades"],
                    "n_wins": n_wins,
                    "n_losses": n_losses,
                    "winrate": metrics["winrate"],
                    "cumulative_return": metrics["cumulative_return"],
                    "avg_r_multiple": metrics["avg_r_multiple"],
                    "worst_trade_drawdown": metrics["worst_trade_drawdown"],
                    "avg_trade_drawdown": metrics["avg_trade_drawdown"],
                    "asset_max_drawdown": metrics["asset_max_drawdown"],
                    "avg_entry_distance": metrics["avg_entry_distance"],
                    "sharpe_ratio": metrics["sharpe_ratio"],
                })

                print(
                    f"{metrics['n_signals_raw']} raw -> {metrics['n_signals']} filtered, "
                    f"{metrics['n_trades']} trades ({n_wins}W/{n_losses}L), "
                    f"WR={metrics['winrate']:.0%}, "
                    f"CR={metrics['cumulative_return']:+.1%}, "
                    f"Sharpe={metrics['sharpe_ratio']:.2f}"
                )

        summary_df = pd.DataFrame(ticker_summaries)
        summary_df["volume_ratio_threshold"] = threshold
        all_summaries.append(summary_df)

        tickers_with_trades = summary_df[summary_df["n_trades"] > 0]

        agg_metrics = {
            "total_signals_raw": int(summary_df["n_signals_raw"].sum()),
            "total_signals_filtered": int(summary_df["n_signals"].sum()),
            "total_filtered_by_template": int(summary_df["n_filtered"].sum()),
            "total_patterns": int(summary_df["n_patterns"].sum()),
            "total_trades": int(summary_df["n_trades"].sum()),
            "total_wins": int(summary_df["n_wins"].sum()),
            "total_losses": int(summary_df["n_losses"].sum()),
            "tickers_with_patterns": int((summary_df["n_patterns"] > 0).sum()),
            "tickers_with_trades": int((summary_df["n_trades"] > 0).sum()),
            "avg_winrate": float(tickers_with_trades["winrate"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
            "median_winrate": float(tickers_with_trades["winrate"].median())
            if len(tickers_with_trades) > 0 else 0.0,
            "avg_cumulative_return": float(tickers_with_trades["cumulative_return"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
            "avg_r_multiple": float(tickers_with_trades["avg_r_multiple"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
            "worst_trade_drawdown": float(tickers_with_trades["worst_trade_drawdown"].min())
            if len(tickers_with_trades) > 0 else 0.0,
            "avg_trade_drawdown": float(tickers_with_trades["avg_trade_drawdown"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
            "avg_entry_distance": float(tickers_with_trades["avg_entry_distance"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
            "avg_sharpe_ratio": float(tickers_with_trades["sharpe_ratio"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
        }
        mlflow.log_metrics(agg_metrics)

        with tempfile.TemporaryDirectory() as tmpdir:
            summary_path = Path(tmpdir) / f"summary_{config_name}.csv"
            summary_df.to_csv(summary_path, index=False)
            mlflow.log_artifact(str(summary_path), "summaries")

            if all_trades_for_config:
                all_trades_df = pd.concat(all_trades_for_config, ignore_index=True)
                all_trades_path = Path(tmpdir) / f"all_trades_{config_name}.csv"
                all_trades_df.to_csv(all_trades_path, index=False)
                mlflow.log_artifact(str(all_trades_path), "summaries")

            # --- Tabla de performance: retorno acumulado, win-rate y sharpe por ticker ---
            perf_cols = ["ticker", "n_trades", "winrate", "cumulative_return", "avg_r_multiple", "sharpe_ratio"]
            perf_df = summary_df[perf_cols].copy()
            perf_df.columns = ["ticker", "n_trades", "win_rate", "cumulative_return", "avg_r_multiple", "sharpe_ratio"]
            totals = pd.DataFrame([{
                "ticker": "TOTAL",
                "n_trades": int(perf_df["n_trades"].sum()),
                "win_rate": float(perf_df.loc[perf_df["n_trades"] > 0, "win_rate"].mean()) if (perf_df["n_trades"] > 0).any() else 0.0,
                "cumulative_return": float(perf_df.loc[perf_df["n_trades"] > 0, "cumulative_return"].mean()) if (perf_df["n_trades"] > 0).any() else 0.0,
                "avg_r_multiple": float(perf_df.loc[perf_df["n_trades"] > 0, "avg_r_multiple"].mean()) if (perf_df["n_trades"] > 0).any() else 0.0,
                "sharpe_ratio": float(perf_df.loc[perf_df["n_trades"] > 0, "sharpe_ratio"].mean()) if (perf_df["n_trades"] > 0).any() else 0.0,
            }])
            perf_df = pd.concat([perf_df, totals], ignore_index=True)
            perf_path = Path(tmpdir) / f"performance_{config_name}.csv"
            perf_df.to_csv(perf_path, index=False)
            mlflow.log_artifact(str(perf_path), "summaries")

            # --- Tabla de trades por ticker: entradas, wins, losses ---
            breakdown_cols = ["ticker", "n_trades", "n_wins", "n_losses", "winrate"]
            breakdown_df = summary_df[breakdown_cols].copy()
            breakdown_df.columns = ["ticker", "n_trades", "n_wins", "n_losses", "win_rate"]
            totals_bd = pd.DataFrame([{
                "ticker": "TOTAL",
                "n_trades": int(breakdown_df["n_trades"].sum()),
                "n_wins": int(breakdown_df["n_wins"].sum()),
                "n_losses": int(breakdown_df["n_losses"].sum()),
                "win_rate": int(breakdown_df["n_wins"].sum()) / max(int(breakdown_df["n_trades"].sum()), 1),
            }])
            breakdown_df = pd.concat([breakdown_df, totals_bd], ignore_index=True)
            breakdown_path = Path(tmpdir) / f"trades_breakdown_{config_name}.csv"
            breakdown_df.to_csv(breakdown_path, index=False)
            mlflow.log_artifact(str(breakdown_path), "summaries")

        print(
            f"\n  AGREGADO: {agg_metrics['total_signals_raw']} raw, "
            f"{agg_metrics['total_filtered_by_template']} filtradas template, "
            f"{agg_metrics['total_trades']} trades ({agg_metrics['total_wins']}W/{agg_metrics['total_losses']}L), "
            f"avg WR={agg_metrics['avg_winrate']:.0%}, "
            f"avg CR={agg_metrics['avg_cumulative_return']:+.1%}, "
            f"avg Sharpe={agg_metrics['avg_sharpe_ratio']:.2f}"
        )

print("\nExperimento completo!")

## 6. Comparacion entre configuraciones

In [19]:
comparison_df = pd.concat(all_summaries, ignore_index=True)

# --- Senales: Raw vs Filtradas ---
if USE_TREND_TEMPLATE:
    ref_threshold = 1.5 if 1.5 in comparison_df["volume_ratio_threshold"].values else comparison_df["volume_ratio_threshold"].iloc[0]
    ref = comparison_df[comparison_df["volume_ratio_threshold"] == ref_threshold]
    filter_comparison = pd.DataFrame({
        "raw": ref.set_index("ticker")["n_signals_raw"],
        "con_filtros": ref.set_index("ticker")["n_signals"],
    })
    filter_comparison["descartadas"] = filter_comparison["raw"] - filter_comparison["con_filtros"]
    filter_comparison["pct_descartadas"] = (
        filter_comparison["descartadas"] / filter_comparison["raw"].replace(0, np.nan)
    )
    print(f"=== Senales VCP: Raw vs Filtradas (threshold={ref_threshold}) ===")
    display(filter_comparison.style.format({
        "raw": "{:.0f}", "con_filtros": "{:.0f}", "descartadas": "{:.0f}",
        "pct_descartadas": "{:.0%}",
    }))
    total_raw = filter_comparison["raw"].sum()
    total_kept = filter_comparison["con_filtros"].sum()
    print(f"\nTotal: {total_raw:.0f} raw -> {total_kept:.0f} post-filtro ({total_kept/max(total_raw,1):.0%} retenidas)")

# --- Trades Breakdown: entradas, wins, losses por ticker y threshold ---
for thresh in VOLUME_RATIO_THRESHOLDS:
    print(f"\n=== Trades Breakdown (threshold={thresh}) ===")
    bd = comparison_df[comparison_df["volume_ratio_threshold"] == thresh][
        ["ticker", "n_trades", "n_wins", "n_losses", "winrate"]
    ].copy()
    bd.columns = ["ticker", "trades", "wins", "losses", "win_rate"]
    bd = bd.set_index("ticker")
    totals_row = pd.DataFrame([{
        "trades": int(bd["trades"].sum()),
        "wins": int(bd["wins"].sum()),
        "losses": int(bd["losses"].sum()),
        "win_rate": int(bd["wins"].sum()) / max(int(bd["trades"].sum()), 1),
    }], index=["TOTAL"])
    bd = pd.concat([bd, totals_row])
    display(bd.style.format({
        "trades": "{:.0f}", "wins": "{:.0f}", "losses": "{:.0f}",
        "win_rate": "{:.0%}",
    }).background_gradient(subset=["win_rate"], cmap="RdYlGn", vmin=0, vmax=1))

# --- Sharpe Ratio (in-trade) ---
pivot_sharpe = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="sharpe_ratio")
print("\n=== Sharpe Ratio (in-trade, anualizado) por Ticker y Threshold ===")

def color_sharpe(val):
    if isinstance(val, (int, float)) and not np.isnan(val):
        if val >= 1.0:
            return "background-color: #27ae60; color: white"
        elif val >= 0.0:
            return "background-color: #f9e79f"
        else:
            return "background-color: #e74c3c; color: white"
    return ""

display(pivot_sharpe.style.format("{:.2f}").map(color_sharpe))

# --- Win Rate ---
pivot_winrate = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="winrate")
print("\n=== Win Rate por Ticker y Threshold ===")
display(pivot_winrate.style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

# --- Retorno Acumulado ---
pivot_cumret = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="cumulative_return")
print("\n=== Retorno Acumulado por Ticker y Threshold ===")

def color_returns(val):
    if isinstance(val, (int, float)) and not np.isnan(val):
        if val > 0:
            return "background-color: #27ae60; color: white"
        elif val < 0:
            return "background-color: #e74c3c; color: white"
    return ""

display(pivot_cumret.style.format("{:+.1%}").map(color_returns))

# --- Drawdown por Trade (peor) ---
pivot_worst_dd = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="worst_trade_drawdown")
print("\n=== Peor Drawdown por Trade ===")
display(pivot_worst_dd.style.format("{:+.1%}").background_gradient(cmap="RdYlGn", vmin=-0.3, vmax=0))

# --- Drawdown promedio por Trade ---
pivot_avg_dd = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="avg_trade_drawdown")
print("\n=== Drawdown Promedio por Trade ===")
display(pivot_avg_dd.style.format("{:+.1%}").background_gradient(cmap="RdYlGn", vmin=-0.2, vmax=0))

# --- Entry Distance promedio ---
pivot_entry_dist = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="avg_entry_distance")
print("\n=== Entry Distance Promedio (% sobre pivote) ===")
display(pivot_entry_dist.style.format("{:.2%}").background_gradient(cmap="YlOrRd", vmin=0, vmax=0.05))

# --- Patrones Detectados ---
pivot_patterns = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="n_patterns")
print("\n=== Patrones Detectados ===")
display(pivot_patterns.style.format("{:.0f}").background_gradient(cmap="Blues"))


=== Trades Breakdown (threshold=1.0) ===


,trades,wins,losses,win_rate
AAPL,7,6,1,86%
AMZN,18,7,11,39%
AVGO,6,4,2,67%
BRK.B,6,2,4,33%
COIN,3,0,3,0%
GLD,5,3,2,60%
GOOGL,11,4,7,36%
HOOD,2,0,2,0%
IWM,7,4,3,57%
JPM,7,3,4,43%



=== Trades Breakdown (threshold=1.5) ===


,trades,wins,losses,win_rate
AAPL,6,5,1,83%
AMZN,14,5,9,36%
AVGO,5,2,3,40%
BRK.B,3,1,2,33%
COIN,1,0,1,0%
GLD,5,3,2,60%
GOOGL,9,3,6,33%
HOOD,1,0,1,0%
IWM,2,1,1,50%
JPM,6,4,2,67%



=== Trades Breakdown (threshold=2.0) ===


...

volume_ratio_threshold,1.000000,1.500000,2.000000
ticker,,,
AAPL,-6.0%,-5.2%,-3.8%
AMZN,-7.0%,-7.7%,-7.2%
AVGO,-8.0%,-7.8%,-7.7%
BRK.B,-3.8%,-3.5%,+0.0%
COIN,-19.6%,-11.8%,+0.0%
GLD,-4.1%,-3.4%,-3.7%
GOOGL,-5.1%,-5.4%,-5.0%
HOOD,-14.2%,-15.9%,-15.9%
IWM,-3.7%,-3.3%,-5.5%



=== Entry Distance Promedio (% sobre pivote) ===


volume_ratio_threshold,1.000000,1.500000,2.000000
ticker,,,
AAPL,0.00%,0.00%,0.00%
AMZN,0.00%,0.00%,0.00%
AVGO,0.00%,0.00%,0.00%
BRK.B,0.00%,0.00%,0.00%
COIN,0.00%,0.00%,0.00%
GLD,0.00%,0.00%,0.00%
GOOGL,0.00%,0.00%,0.00%
HOOD,0.00%,0.00%,0.00%
IWM,0.00%,0.00%,0.00%



=== Patrones Detectados ===


volume_ratio_threshold,1.000000,1.500000,2.000000
ticker,,,
AAPL,7,6,3
AMZN,18,14,7
AVGO,6,5,3
BRK.B,6,3,0
COIN,3,1,0
GLD,5,5,3
GOOGL,11,9,3
HOOD,2,1,1
IWM,7,2,1


## 7. Grafico comparativo agregado

In [ ]:
matplotlib.use("Agg")

traded_df = comparison_df[comparison_df["n_trades"] > 0]

agg_by_threshold = traded_df.groupby("volume_ratio_threshold").agg({
    "n_patterns": "sum",
    "n_trades": "sum",
    "winrate": "mean",
    "cumulative_return": "mean",
    "worst_trade_drawdown": "min",
    "avg_trade_drawdown": "mean",
    "avg_entry_distance": "mean",
}).reset_index()

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

plot_configs = [
    ("n_patterns", "Total Patrones", "{:.0f}"),
    ("n_trades", "Total Trades", "{:.0f}"),
    ("winrate", "Avg Win Rate", "{:.0%}"),
    ("cumulative_return", "Avg Retorno Acum.", "{:+.1%}"),
    ("worst_trade_drawdown", "Peor DD por Trade", "{:+.1%}"),
    ("avg_trade_drawdown", "Avg DD por Trade", "{:+.1%}"),
    ("avg_entry_distance", "Avg Entry Distance", "{:.2%}"),
]

colors = ["#3498db", "#e67e22", "#27ae60", "#9b59b6"]

for ax, (col, title, fmt) in zip(axes, plot_configs):
    bars = ax.bar(
        agg_by_threshold["volume_ratio_threshold"].astype(str),
        agg_by_threshold[col],
        color=colors[:len(agg_by_threshold)],
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("volume_ratio_threshold")
    for bar, val in zip(bars, agg_by_threshold[col]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            fmt.format(val),
            ha="center",
            va="bottom" if val >= 0 else "top",
            fontsize=10,
        )

# Ocultar el 8vo subplot sobrante
axes[-1].set_visible(False)

filters_active = []
if USE_TREND_TEMPLATE:
    filters_active.append("TrendTemplate")
if SEQUENCE_PARAMS["require_ascending_lows"]:
    filters_active.append("AscLows")
if BREAKOUT_PARAMS["max_entry_distance_pct"] is not None:
    filters_active.append(f"MaxDist={BREAKOUT_PARAMS['max_entry_distance_pct']:.0%}")
if USE_VOLUME_CONTRACTION:
    filters_active.append("VolContraction")
filters_str = ", ".join(filters_active) if filters_active else "Ninguno"

plt.suptitle(
    f"VCP Full Filters - Comparacion por Threshold\nFiltros: {filters_str}",
    fontsize=14, fontweight="bold", y=1.02,
)
plt.tight_layout()
chart_path = project_root / "experiments" / "full_filters_comparison.png"
fig.savefig(str(chart_path), dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Grafico guardado en: {chart_path}")

display(agg_by_threshold.style.format({
    "n_patterns": "{:.0f}",
    "n_trades": "{:.0f}",
    "winrate": "{:.0%}",
    "cumulative_return": "{:+.1%}",
    "worst_trade_drawdown": "{:+.1%}",
    "avg_trade_drawdown": "{:+.1%}",
    "avg_entry_distance": "{:.2%}",
}))

## 8. Resumen: impacto de los filtros

In [ ]:
total_by_threshold = comparison_df.groupby("volume_ratio_threshold").agg({
    "n_signals_raw": "sum",
    "n_signals": "sum",
    "n_filtered": "sum",
    "n_trades": "sum",
}).reset_index()

total_by_threshold["pct_retained"] = (
    total_by_threshold["n_signals"] / total_by_threshold["n_signals_raw"].replace(0, np.nan)
)

print("=== Impacto de Filtros por Threshold ===\n")
display(total_by_threshold.rename(columns={
    "n_signals_raw": "sin_filtro",
    "n_signals": "con_filtros",
    "n_filtered": "descartadas_template",
    "n_trades": "trades_finales",
    "pct_retained": "pct_retenidas",
}).style.format({
    "sin_filtro": "{:.0f}",
    "con_filtros": "{:.0f}",
    "descartadas_template": "{:.0f}",
    "trades_finales": "{:.0f}",
    "pct_retenidas": "{:.1%}",
}))

# --- Resumen de configuracion ---
print("\n=== Configuracion de Filtros ===")
print(f"  Trend Template:        {'ON' if USE_TREND_TEMPLATE else 'OFF'}")
print(f"  Ascending Lows:        {'ON' if SEQUENCE_PARAMS['require_ascending_lows'] else 'OFF'} (tol={SEQUENCE_PARAMS['ascending_lows_tolerance']})")
print(f"  Max Entry Distance:    {BREAKOUT_PARAMS['max_entry_distance_pct'] if BREAKOUT_PARAMS['max_entry_distance_pct'] is not None else 'OFF'}")
print(f"  Volume Confirmation:   {'ON' if BREAKOUT_PARAMS['require_volume_confirmation'] else 'OFF'}")
print(f"  Volume Contraction:    {'ON' if USE_VOLUME_CONTRACTION else 'OFF'}")
print(f"  Max Depth:             {SEQUENCE_PARAMS['max_depth_pct'] if SEQUENCE_PARAMS['max_depth_pct'] is not None else 'OFF'}")
print(f"  Min Total Reduction:   {SEQUENCE_PARAMS['min_total_reduction'] if SEQUENCE_PARAMS['min_total_reduction'] is not None else 'OFF'}")

# --- Resumen global ---
print("\n=== Resumen Global ===")
total_raw = int(comparison_df["n_signals_raw"].sum())
total_kept = int(comparison_df["n_signals"].sum())
total_trades = int(comparison_df["n_trades"].sum())
total_discarded = int(comparison_df["n_filtered"].sum())
print(f"Senales VCP totales (pre-filtros):     {total_raw}")
print(f"Senales retenidas (post-filtros):      {total_kept} ({total_kept/max(total_raw,1):.1%})")
if USE_TREND_TEMPLATE:
    print(f"Descartadas por Trend Template:        {total_discarded}")
print(f"Trades ejecutados:                     {total_trades}")

## 9. Conclusiones

Para explorar los resultados en detalle:
```bash
mlflow ui --backend-store-uri mlruns/
```

### Filtros implementados

1. **Trend Template** (`USE_TREND_TEMPLATE`): valida 7 condiciones de Etapa 2 de Minervini.
   Descarta senales en activos fuera de fase alcista.

2. **Ascending Lows** (`require_ascending_lows`): exige que los lows de cada contraccion
   sean ascendentes. Confirma que compradores entran a niveles cada vez mas altos.

3. **Max Entry Distance** (`max_entry_distance_pct`): rechaza breakouts donde el precio
   ya se alejo demasiado del pivote (risk/reward desfavorable).

4. **Volume Contraction** (`USE_VOLUME_CONTRACTION`): verifica que el volumen se contrae
   durante la formacion del patron.

5. **Quality filters** (`max_depth_pct`, `max_depth_atr`, `min_total_reduction`):
   - `max_depth_pct`: profundidad maxima porcentual por contraccion individual (absoluto).
   - `max_depth_atr`: profundidad maxima en multiplos de ATR por contraccion (relativo a
     volatilidad). Normaliza el filtro entre activos de distinta volatilidad: una caida de
     15% es excesiva para AAPL (ATR ~1.5%, depth_atr=10) pero normal para MELI (ATR ~4%,
     depth_atr=3.7). Ambos filtros se aplican en conjunto.
   - `min_total_reduction`: compresion minima total de la secuencia.

### Como desactivar filtros

Para comparar el impacto de cada filtro, desactivar uno a la vez en la celda de
configuracion y re-ejecutar. Los parametros de MLflow registran el estado de cada filtro.